In [1]:
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)
from utils.llm_word_logprobs import *

import mne

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator
import seaborn as sns

import math
import re
from collections import namedtuple
from typing import Callable

from sklearn.model_selection import LeaveOneOut, KFold
from sklearn.linear_model import Ridge
import torch
from transformers import BertTokenizerFast, AutoModelForCausalLM

from scipy.stats import ttest_1samp
from statsmodels.stats.multitest import fdrcorrection

from IPython.display import clear_output
import warnings

/Users/jowanglin/regression-based_ERP/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Deconvolution V2

In [2]:
DATA_DIR = "/Users/jowanglin/regression-based_ERP/data/eeg/crystal"
STIMULI_EXCEL = "/Users/jowanglin/regression-based_ERP/data/stimuli/CRYSTAL_master-sheet.xlsx"

SOA = 0.36     
DOWNSAMPLE_SFREQ = 100.0  
N_WORDS = 7
N_TRIALS = 208

RIDGE_TOL = 1e-5
MAX_ITER = 20000
RANDOM_STATE = 42

### Get word log probabilities
Following [CKIP Lab documentation](https://github.com/ckiplab/ckip-transformers):
- Tokenizer should be BERT `BertTokenizerFast.from_pretrained("bert-base-chinese")`
- Use a causal LLM, e.g., `AutoModelForCausalLM.from_pretrained("ckiplab/gpt2-base-chinese") `
- `is_split_into_words=False`, so token-to-word alignment is manually done inside the function `get_word_logprobs`, rather than relying on the tokenizer's offset mapping.

In [3]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-chinese")
model = AutoModelForCausalLM.from_pretrained("ckiplab/gpt2-base-chinese") 

Loading weights: 100%|██████████| 149/149 [00:00<00:00, 31075.10it/s]
[transformers] GPT2LMHeadModel LOAD REPORT from: ckiplab/gpt2-base-chinese
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
file_dir = "/Users/jowanglin/regression-based_ERP/data/stimuli"
file_name = "CRYSTAL_master-sheet.xlsx"
sheet_name = "Overall"
frame_col = "Sentence_frame"
word_cols = [f"W{i}" for i in range(1,13)]
group_by = "List"
prompt = "通順嗎？"

sent_and_splits_list = get_sentences_and_splits(file_dir,
                                                file_name,
                                                sheet_name,
                                                frame_col=frame_col,
                                                word_cols=word_cols,
                                                group_by=group_by,
                                                prompt=prompt)

pad_token, cls_token, sep_token = tokenizer.pad_token, tokenizer.cls_token, tokenizer.sep_token
pad_id, cls_id, sep_id = tokenizer.pad_token_id, tokenizer.cls_token_id, tokenizer.sep_token_id

from_llm = {}
for i, (sentences, split_sentences) in enumerate(sent_and_splits_list):
    ids, tokens, token_logprobs = get_llm_token_logprobs(sentences,
                                                         model, 
                                                         tokenizer,
                                                         max_length=MAX_LENGTH)
    word_logprobs = get_word_logprobs(tokens=tokens,
                                  split_sentences=split_sentences,
                                  ids=ids,
                                  token_logprobs=token_logprobs,
                                  special_tokens=(pad_token, cls_token, sep_token),
                                  special_token_ids=(pad_id, cls_id, sep_id),
                                  other_tokens=OTHER_TOKENS)
    from_llm[i+1] = {"ids": ids,
                     "tokens": tokens,
                     "token_logprobs": token_logprobs,
                     "word_logprobs": word_logprobs}



[UNK] found in sentence index 21, token index 24:
  ['她', '家', '就', '在', '學', '校', '旁', '還', '每', '次', '遲', '到', '就', '說', '因', '為', '塞', '車', '，', '其', '實', '都', '是', '胡', '[UNK]', '。'] 
  ['她家', '就在', '學校旁', '還每次', '遲到', '就說', '因為', '塞車，', '其實', '都是', '胡謅。']
>> Sentence indices where alignment failed: [21]

[UNK] found in sentence index 32, token index 21:
  ['去', '完', '角', '質', '塗', '上', '乳', '液', '後', '，', '皮', '膚', '摸', '起', '來', '像', '打', '蠟', '一', '般', '胡', '[UNK]', '。'] 
  ['去完', '角質', '塗上', '乳液後，', '皮膚', '摸起來', '像打蠟', '一般', '胡謅。']
>> Sentence indices where alignment failed: [32]



In [5]:
def inspect_probs(tokenizer, from_llm: dict, which_list: int, row: int, to_prob=False):
    ids = from_llm[which_list]["ids"]
    token_logprobs = from_llm[which_list]["token_logprobs"]
    tok_row = tokenizer.convert_ids_to_tokens(ids[row])
    p_row = token_logprobs[row] 
    if to_prob:
        p_row = np.exp(p_row)
        p_row = [f"{x:.6f}" for x in p_row]
        prob_col = "prob"
    else:
        prob_col = "logprob"
    
    df = pd.DataFrame({"tokens": tok_row, prob_col: p_row})
    print(f"Row: {row}")
    print(df)

inspect_probs(tokenizer, from_llm, which_list=1, row=100, to_prob=True)


Row: 100
   tokens      prob
0   [CLS]  1.000000
1       她  0.000500
2       拿  0.000262
3       別  0.000301
4       人  0.794731
5       的  0.243271
6       計  0.000690
7       畫  0.307847
8       去  0.006718
9       發  0.008184
10      表  0.045123
11      ，  0.035010
12      被  0.004046
13      抓  0.002407
14      到  0.049313
15      還  0.000633
16      不  0.051038
17      承  0.000658
18      認  0.962679
19      實  0.000516
20      在  0.028878
21      不  0.070107
22      知  0.018739
23      習  0.000059
24      慣  0.010802
25      。  0.174250
26  [SEP]  0.000000
27  [PAD]  0.000000
28  [PAD]  0.000000


In [14]:
list1_and_list2_word_logprobs = {1:from_llm[1]["word_logprobs"],
                                 2:from_llm[2]["word_logprobs"]}

for i, (word_logprobs, (sentences, split_sentences)) in enumerate(zip(list(list1_and_list2_word_logprobs.values()),
                                                                      sent_and_splits_list)):
    print(f"STIMULI LIST {i+1}")
    word_counts = [len(s) for s in split_sentences]
    word_logprobs_counts = [len(p) for p in word_logprobs]
    not_aligned = np.where(np.array(word_counts) - np.array(word_logprobs_counts) != 0)[0]
    print(f"  len(word_logprobs) = {len(word_logprobs)}")
    print(f"  word_logprobs indecides where get_sentences_and_splits failed to align: {not_aligned}")
    print(f"  each row of word_logprobs has >= {N_WORDS} words: {np.all(np.array(word_logprobs_counts) >= N_WORDS)}\n")

STIMULI LIST 1
  len(word_logprobs) = 208
  word_logprobs indecides where get_sentences_and_splits failed to align: [21]
  each row of word_logprobs has >= 7 words: True

STIMULI LIST 2
  len(word_logprobs) = 208
  word_logprobs indecides where get_sentences_and_splits failed to align: [32]
  each row of word_logprobs has >= 7 words: True



### Build design matrix
- `build_design` now supports any number of predictors and aautomatically adds the intercept if `intercept=True`

In [15]:
def build_design(*, soa: float, sfreq: float,
                 n_words: int, n_trials: int,
                 response_lag: float,
                 rank_tol: float=1e-3,
                 intercept: bool=True,
                 **predictors):
    if predictors:
        for p in predictors.values():
            assert p.shape == (n_trials, n_words), "Each predictor must be a numpy array of shape (n_trials, n_words)"

    DeconvDesignMatrix = namedtuple("DeconvDesignMatrix",
                                    ["predictors", "full", "n_lags", "rank", "singular_vals", "condition_num"])
    
    predictors = dict(predictors)
    if intercept:
        # predictors["intercept"] = np.ones((n_trials, n_words), dtype=float)
        predictors = {"intercept": np.ones((n_trials, n_words), dtype=float), **predictors}

    n_time_samples_per_trial = int(round(soa * n_words * sfreq))
    n_time_samples = int(n_time_samples_per_trial * n_trials) 
    n_lags = int(round(response_lag * sfreq))
    soa_samples = int(round(soa * sfreq))
    
    matrices = {name: np.zeros((n_time_samples, n_lags), dtype=float) for name in predictors}
    
    positions = range(n_words)
    for i in range(n_trials):
        for j in positions:
            valid_range = min(n_time_samples_per_trial - j * soa_samples, n_lags)
            row = i * n_time_samples_per_trial + j * soa_samples
            for m, p in zip(matrices.values(), predictors.values()):
                m[row : row + valid_range, : valid_range] += p[i, j] * np.eye(valid_range)

    X_full = np.hstack([m for m in matrices.values()])
    rank = np.linalg.matrix_rank(X_full, tol=rank_tol)
    singular_vals = np.linalg.svd(X_full, compute_uv=False)
    condition_num = np.linalg.cond(X_full)

    return DeconvDesignMatrix(matrices, X_full, n_lags, rank, singular_vals, condition_num)




In [16]:
word_pos_predictor = np.vstack([np.log(np.arange(1, N_WORDS+1)) for _ in range(N_TRIALS)])

list1_suprisal = - np.stack([p[:N_WORDS] for p in list1_and_list2_word_logprobs[1]])
list2_suprisal = - np.stack([p[:N_WORDS] for p in list1_and_list2_word_logprobs[2]])
assert np.isfinite(list1_suprisal).all()
assert np.isfinite(list2_suprisal).all()

surprisal_z = np.stack([list1_suprisal, list2_suprisal], axis=0)
surprisal_z = (surprisal_z - np.mean(surprisal_z)) / np.std(surprisal_z)
list1_suprisal = surprisal_z[0]
list2_suprisal = surprisal_z[1]

list1_matrix = build_design(soa=SOA,
                            sfreq=DOWNSAMPLE_SFREQ,
                            n_words=N_WORDS, n_trials=N_TRIALS,
                            response_lag=0.7,
                            rank_tol=1e-2,
                            intercept=True,
                            word_pos=word_pos_predictor,
                            surprisal=list1_suprisal)
list2_matrix = build_design(soa=SOA,
                            sfreq=DOWNSAMPLE_SFREQ,
                            n_words=N_WORDS, n_trials=N_TRIALS,
                            response_lag=0.7,
                            rank_tol=1e-2,
                            intercept=True,
                            word_pos=word_pos_predictor,
                            surprisal=list2_suprisal)

for i, m in enumerate((list1_matrix, list2_matrix)):
    print(f"LIST {i+1}")
    print(f"X_full.shape = {m.full.shape}")
    print(f"rank = {m.rank}")
    print(f"condition number = {m.condition_num}")
    print(f"5 largest singular values: {m.singular_vals[:5]}")
    print(f"5 smallest singular values: {m.singular_vals[-5:]}\n")


LIST 1
X_full.shape = (52416, 210)
rank = 210
condition number = 101.2335638081273
5 largest singular values: [83.43972025 83.43972025 83.43972025 83.43972025 83.43972025]
5 smallest singular values: [0.8242298 0.8242298 0.8242298 0.8242298 0.8242298]

LIST 2
X_full.shape = (52416, 210)
rank = 210
condition number = 101.2482866516649
5 largest singular values: [83.44320422 83.44320422 83.44320422 83.44320422 83.44320422]
5 smallest singular values: [0.82414436 0.82414436 0.82414436 0.82414436 0.82414436]



### Deconvolution with SSP
- Create custom class `MySSP`
- `tune_alpha` and `nested_cv` now support non-aggregated trial scoring is `score_on_mean=False`. If `score_on_mean=True`, the time-lagged design matrix will be averaged across trials (when the design matrix is not trial-specific, e.g., predictors are intercept and word position only, this is equivalent to a single-trial design matrix).
- If `ssp=True`, the input data to the model should contain all channels including EOGs for the SSP estimation to be meaningful; however, `roi_idx` must be specified (and align with the channel indices of `instance.ch_names`) as the hyperparameter (alpha) is selected based on the scores on ROI channels only.

In [17]:
class MySSP:
    # TODO: probably make it inherit from sklearn preprocessor or something
    # for better integration for future Amanda :D
    def __init__(self): 
        pass
    def fit_transform(self):
        pass
    def transform(self):
        pass


def tune_alpha(ridge_data: np.ndarray, *, X: np.ndarray,  
               alpha_range: np.ndarray,
               ssp: bool=False, info: mne.Info | None=None,
               roi_idx: list | None=None,
               loo: bool=False, n_splits: int | None=None,
               ridge_tol: float=1e-4, solver: str="auto", max_iter: int|None=None,
               score_on_mean: bool=True,
               verbose: bool=False):
    """ridge_data.shape = (n_trials, n_channels, n_time_samples); n_time_samples is total, not per trial
                          if doing LOO (e.g., training on 19 subjects, testing on 1), n_trials = n_subjects (but this does not work well)
       design matrix X.shape = (n_time_samples, n_predictors * n_lags)
       alpha_range in single-subject-level range (not divided by n_subj-1)
       if ssp is True, ridge_data should contain the full channel set needed for SSP, then only later restrict reporting / inference
       to ROI channels, where roi_idx must refer to the channel indices of the original full channel set.
       (If ridge_data contains only the ROI channels, then SSP is not very meaningful.)"""
    
    ScoreResults = namedtuple("ScoreResults", ["train_scores", "train_mean", "train_std",
                                               "valid_scores", "valid_mean", "valid_std"])
    CVResults = namedtuple("CVResults", ["fold_scores", "best_mean_score", "best_alpha"])   
    
    n_trials, n_channels, n_time_samples_per_trial = ridge_data.shape
    assert n_trials * n_time_samples_per_trial  == X.shape[0]
    X_split = np.stack(np.split(X, n_trials), axis=0)    # stack into an array so advance indexing works for train_idx and valid_idx
                                                         # X_split.shape (n_trials, n_time_samples_per_trial, n_lags)
    if roi_idx is None:
        print(f"ROI channel indices not set. Default to all channels in the input data.")
        roi_idx = list(range(n_channels))

    if loo:
        splitter = LeaveOneOut()
        n_splits = n_trials
    else:
        if n_splits is None:
            n_splits = 5
        splitter = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    
    train_scores = np.empty((n_splits, len(alpha_range), len(roi_idx)))
    valid_scores = np.empty((n_splits, len(alpha_range), len(roi_idx)))

    for i, (train_idx, valid_idx) in enumerate(splitter.split(ridge_data)):
        if verbose:
            print(f"  FOLD {i+1}")
         
        X_train_stack = np.vstack(X_split[train_idx])
        X_valid_stack = np.vstack(X_split[valid_idx])

        if score_on_mean:
            X_train_mean = np.mean(X_split[train_idx], axis=0)
            X_valid_mean = np.mean(X_split[valid_idx], axis=0)

        y_train, y_valid = ridge_data[train_idx], ridge_data[valid_idx]  # y_train.shape = (n_train_trials, n_channels, n_time_samples_per_trial)  
        y_train_stack = np.hstack([y for y in y_train])    # shape = (n_channels, n_train_trials * n_time_samples_per_trial)
        y_valid_stack = np.hstack([y for y in y_valid])    # shape = (n_channels, n_valid_trials * n_time_samples_per_trial)

        if ssp:
            my_ssp = MySSP(info=info)
            y_train_stack = my_ssp.fit_transform(y_train_stack)       
            y_valid_stack = my_ssp.transform(y_valid_stack)       

            y_train = np.stack(np.split(y_train_stack, len(train_idx), axis=1), axis=0)
            y_valid = np.stack(np.split(y_valid_stack, len(valid_idx), axis=1), axis=0)
            # y_train = y_train_stack.reshape(y_train.shape)  <- not safe
            # y_valid = y_valid_stack.reshape(y_valid.shape)  <- not safe    
        
        for j, alpha in enumerate(alpha_range):
            if verbose:
                print(f"ALPHA = {alpha}")   

            ridge = Ridge(alpha=alpha,
                          fit_intercept=False,         
                          tol=ridge_tol,
                          solver=solver,
                          positive=False,              
                          max_iter=max_iter,
                          random_state=RANDOM_STATE)
            
            for k, ch in enumerate(roi_idx):
                if verbose:
                    print(f"    Channel Index {ch}")

                ridge.fit(X_train_stack, y_train_stack[ch, :])

                if score_on_mean:
                    #X_train_mean = np.mean(X_split[train_idx], axis=0)  <- move outside of channel loop
                    #X_valid_mean = np.mean(X_split[valid_idx], axis=0)  <- move outside of channel loop
                    y_train_per_ch_mean = np.mean(y_train[:, ch, :], axis=0)
                    y_valid_per_ch_mean = np.mean(y_valid[:, ch, :], axis=0)
                    train_scores[i, j, k] = ridge.score(X_train_mean,
                                                        y_train_per_ch_mean)
                    valid_scores[i, j, k] = ridge.score(X_valid_mean,
                                                        y_valid_per_ch_mean)
                else:
                    train_scores[i, j, k] = ridge.score(X_train_stack,
                                                        y_train_stack[ch, :])
                    valid_scores[i, j, k] = ridge.score(X_valid_stack,
                                                        y_valid_stack[ch, :])
    
    fold_scores = {alpha: ScoreResults(train_scores[:, j, :], np.mean(train_scores[:, j, :]), np.std(train_scores[:, j, :]),
                                       valid_scores[:, j, :], np.mean(valid_scores[:, j, :]), np.std(valid_scores[:, j, :]))
                          for j, alpha in enumerate(alpha_range)}
            
    mean_scores = np.array([fold_scores[alpha].valid_mean for alpha in alpha_range])
    best_mean_score = np.max(mean_scores)
    best_alpha = alpha_range[np.argmax(mean_scores)]

    return CVResults(fold_scores, best_mean_score, best_alpha)



In [18]:
def nested_cv(subj_data: np.ndarray, *, X: np.ndarray,
              alpha_range: np.ndarray,
              ssp: bool=False, info: mne.Info | None=None,
              roi_idx: list | None=None,
              n_splits_outer: int=5, n_splits_inner: int=5,
              ridge_tol: float=1e-4, solver: str="auto", max_iter: int|None=None,
              score_on_mean: bool=True,
              verbose: bool=False):
    
    n_trials, n_channels, n_time_samples_per_trial = subj_data.shape
    assert n_trials * n_time_samples_per_trial  == X.shape[0]
    X_split_outer = np.stack(np.split(X, n_trials), axis=0)

    if roi_idx is None and verbose:
        print(f"ROI channel indices not set. Default to all channels in the input data.")
        roi_idx = list(range(n_channels))

    kf = KFold(n_splits=n_splits_outer, shuffle=True, random_state=RANDOM_STATE)

    outer_scores = np.empty((n_splits_outer, len(roi_idx)))
    best_alphas = np.empty((n_splits_outer, ))
    for i, (train_outer_idx, test_outer_idx) in enumerate(kf.split(subj_data)):
        if verbose:
            print(f"OUTER FOLD {i+1}")

        X_train_outer_stack = np.vstack(X_split_outer[train_outer_idx])
        X_test_outer_stack = np.vstack(X_split_outer[test_outer_idx])
        if score_on_mean:
            X_test_outer_mean = np.mean(X_split_outer[test_outer_idx], axis=0)
        
        y_train_outer = subj_data[train_outer_idx]
        y_test_outer = subj_data[test_outer_idx]

        cv_results = tune_alpha(y_train_outer, X=X_train_outer_stack,
                                alpha_range=alpha_range,
                                ssp=ssp, info=info,
                                roi_idx=roi_idx,
                                loo=False,
                                n_splits=n_splits_inner,
                                ridge_tol=ridge_tol,
                                score_on_mean=score_on_mean,
                                max_iter=max_iter,
                                solver=solver,
                                verbose=False)
        best_alphas[i] = cv_results.best_alpha

        ridge = Ridge(alpha=cv_results.best_alpha,
                      fit_intercept=False,         
                      tol=ridge_tol,
                      solver=solver,
                      positive=False,              
                      max_iter=max_iter,
                      random_state=RANDOM_STATE)
        
        y_train_outer_stack = np.hstack([y for y in y_train_outer])
        y_test_outer_stack = np.hstack([y for y in y_test_outer])

        if ssp:
            my_ssp = MySSP(info=info)
            y_train_outer_stack = my_ssp.fit_transform(y_train_outer_stack)
            y_test_outer_stack = my_ssp.transform(y_test_outer_stack)
            # y_train_outer = np.stack(np.split(y_train_outer_stack, len(train_outer_idx), axis=1), axis=0) <- not actually used, leaving it here for debugging
            y_test_outer = np.stack(np.split(y_test_outer_stack, len(test_outer_idx), axis=1), axis=0)

        for j, ch in enumerate(roi_idx):
            ridge.fit(X_train_outer_stack, y_train_outer_stack[ch, :])

            if score_on_mean:
                # X_test_outer_mean = np.mean(X_split_outer[test_outer_idx], axis=0)  <- move outside channel loop
                y_test_outer_per_ch_mean = np.mean(y_test_outer[:, ch, :], axis=0)
                outer_scores[i, j] = ridge.score(X_test_outer_mean,
                                                 y_test_outer_per_ch_mean)
            else:
                outer_scores[i, j] = ridge.score(X_test_outer_stack,
                                                 y_test_outer_stack[ch, :])

    return outer_scores, best_alphas



### Loop over all subjects
- Handles List 1 and List 2 as word logprobs differ between different lists
- Removed `compute_evoked` from the function (not used; ended up manually averaging)

In [30]:
def generate_time_lock_events(position_range: tuple,
                              remove: tuple | list | None=None):
    constraint = "high_constraint", "low_constraint"
    word_positions = [f"w{i}" for i in range(position_range[0], position_range[1]+1)]
    time_lock_events = [f"{w}/{c}" for w in word_positions for c in constraint]
    if remove is not None:
        for r in remove:
            time_lock_events.remove(r)
    return time_lock_events

def revise_annot(df_annot: pd.DataFrame, *,
                 fixation: str, non_final: str, item_codes: list|tuple,
                 high_constraint: tuple, low_constraint: tuple) -> mne.Annotations:   
    desc = list(df_annot["description"])
    desc_revised = desc.copy()
    for i, d in enumerate(desc):
        if d == fixation:
            step = 1
        elif d == non_final:
            desc_revised[i] = "w" + str(step) + "/high_constraint"
            step += 1
        elif d in item_codes:
            if desc[i-1] == fixation:
                desc_revised[i] = "w" + str(step) + "/high_constraint"
                step += 1
            elif desc[i-1] == non_final:
                desc_revised[i] = "w" + str(step) + "/high_constraint"
                #step = 0
        elif d in high_constraint:
            step = 0
        elif d in low_constraint:
            desc_revised[i-step: i] = [s.replace("/high_constraint", "/low_constraint")
                                            for s in desc_revised[i-step: i]]
            step = 0
    
    annot = mne.Annotations(onset=df_annot["onset"], duration=df_annot["duration"], description=desc_revised)
    return annot

def get_segments_of_consecutive_words(raw: mne.io.Raw, *,
                                      fixation: str, non_final: str, item_codes: str,
                                      high_constraint: str, low_constraint: str,
                                      soa: float, n_words: int,
                                      downsample_sfreq: float,
                                      condition_codes: list, list_identifier: dict,
                                      verbose=False):
    
    n_trials = list_identifier[1].shape[0]
    assert list_identifier[2].shape[0] == n_trials, "list_identifier must be a dict of equal-length arrays"
    
    # do NOT use raw.annotations.to_data_frame()
    # I don't know why it doesn't work
    df_annot = pd.DataFrame(raw.annotations)
    conditions = df_annot.loc[df_annot["description"].isin(condition_codes)]["description"].to_numpy()
    if np.array_equal(conditions, list_identifier[1]):
        which_list = 1
    elif np.array_equal(conditions, list_identifier[2]):
        which_list = 2
    else:
        which_list = 0
        length = len(conditions)
        if verbose:
            print(f"Subject trial sequence does not match List 1 or List 2!")
            print(f"Length check: number of trials = {length}")
            if length == n_trials:
                check1 = conditions.astype(int) - list_identifier[1].astype(int)
                check2 = conditions.astype(int) - list_identifier[2].astype(int)
                print(f"Differs from List 1 at {np.where(check1 != 0)[0]}")
                print(f"Differs from List 2 at {np.where(check2 != 0)[0]}")
    
    annot_revised = revise_annot(df_annot,
                                 fixation=fixation, non_final=non_final, item_codes=item_codes,
                                 high_constraint=high_constraint, low_constraint=low_constraint)
    raw = raw.copy()
    raw.set_annotations(annot_revised)

    events, event_id = mne.events_from_annotations(raw)
    
    time_lock_events = generate_time_lock_events(position_range=(1, 1))  # only time lock to word 1
    time_locks = {k: v for k, v in event_id.items() if k in time_lock_events}

    epochs = mne.Epochs(raw,
                        events=events, event_id=time_locks,
                        tmin=0.0, tmax=soa*n_words, baseline=None,  # no baseline correction
                        on_missing="raise",
                        preload=True,
                        verbose=True)
    epochs = epochs.resample(sfreq=downsample_sfreq)
    
    return epochs, which_list

In [31]:
warnings.filterwarnings("ignore", category=RuntimeWarning)

df_stimuli = pd.read_excel(STIMULI_EXCEL, sheet_name="Overall")
fixation = "210"
non_final = "220"
item_codes = df_stimuli["W1_code"].unique().astype(str)
high_constraint = ("240", "241", "242", "243")
low_constraint = ("244", "245", "246", "247")
condition_codes = [str(i) for i in range(240, 248)]

raw1 = mne.io.read_raw_eeglab(f"{DATA_DIR}/subj001_reref_filt.set", preload=False, verbose=False)  # list 1 subj
raw4 = mne.io.read_raw_eeglab(f"{DATA_DIR}/subj004_reref_filt.set", preload=False, verbose=False)  # list 2 subj

list_identifier = {}
for i, r in enumerate((raw1, raw4)):
    df_annot = pd.DataFrame(r.annotations)
    conditions = df_annot.loc[df_annot["description"].isin(condition_codes)]["description"].to_numpy()
    list_identifier[i+1] = conditions

warnings.resetwarnings()

test = revise_annot(raw1.annotations.to_data_frame(),
                    fixation=fixation,
                    non_final=non_final,
                    item_codes=item_codes,
                    high_constraint=high_constraint,
                    low_constraint=low_constraint)
display(test.to_data_frame().iloc[:20])


,onset,duration,description
0,1970-05-23 02:00:00,0.0,1
1,1970-10-27 17:53:20,0.0,210
2,1970-11-09 02:00:00,0.0,w1/high_constraint
3,1970-11-13 06:00:00,0.0,w2/high_constraint
4,1970-11-17 10:00:00,0.0,w3/high_constraint
5,1970-11-21 14:00:00,0.0,w4/high_constraint
6,1970-11-25 18:00:00,0.0,w5/high_constraint
7,1970-11-29 22:00:00,0.0,w6/high_constraint
8,1970-12-04 02:00:00,0.0,w7/high_constraint
9,1970-12-08 06:00:00,0.0,w8/high_constraint


In [32]:
list_of_epochs = []
subj_list_check = {}
for num in range(1, 28):
    raw_file_name = f"subj{str(num).zfill(3)}_reref_filt.set"
    try:
        raw = mne.io.read_raw_eeglab(f"{DATA_DIR}/{raw_file_name}", preload=True, verbose=False)
    except FileNotFoundError:
        continue
    print(f"subj{str(num).zfill(3)}")
    epochs, which_list = get_segments_of_consecutive_words(raw,
                                                          fixation=fixation,
                                                          non_final=non_final, item_codes=item_codes,
                                                          high_constraint=high_constraint, low_constraint=low_constraint,
                                                          soa=SOA, n_words=N_WORDS,
                                                          downsample_sfreq=DOWNSAMPLE_SFREQ,
                                                          condition_codes=condition_codes,
                                                          list_identifier=list_identifier,
                                                          verbose=True)
    list_of_epochs.append((epochs, which_list))
    subj_list_check[f"subj{str(num).zfill(3)}"] = which_list

clear_output()
print(subj_list_check)

list_of_epochs_drop_bad_subj = list(filter(lambda e: e[1] != 0, list_of_epochs))
print(f"number of subjects after filtering out bad stimuli alignment (with either List 1 or List 2's template): {len(list_of_epochs_drop_bad_subj)}")


{'subj001': 1, 'subj003': 1, 'subj004': 2, 'subj005': 2, 'subj006': 1, 'subj007': 2, 'subj008': 0, 'subj009': 2, 'subj010': 1, 'subj011': 2, 'subj012': 2, 'subj014': 2, 'subj017': 1, 'subj018': 2, 'subj020': 2, 'subj021': 1, 'subj022': 2, 'subj024': 1, 'subj025': 1, 'subj027': 1}
number of subjects after filtering out bad stimuli alignment (with either List 1 or List 2's template): 19


### Nested CV per-subject

In [33]:
ch_roi = ["CZ", "C3", "C4", "CP3", "CP4", "CPZ"]
ch_roi_to_idx = {ch: idx for idx, ch in enumerate(list_of_epochs[0][0].ch_names)}
roi_idx = [ch_roi_to_idx[ch] for ch in ch_roi]

alpha_range = np.logspace(-3, 3, 25)  # 0.001 -> 1000

outer_scores_int_logprobs_per_subj = np.empty((len(list_of_epochs_drop_bad_subj), ), dtype=object)
outer_scores_int_per_subj = np.empty((len(list_of_epochs_drop_bad_subj), ), dtype=object)
best_alphas_per_subj = []

X_int = list1_matrix.predictors["intercept"]

for i, (epochs, which_list) in enumerate(list_of_epochs_drop_bad_subj):
    print(f"subject index [{i}]...")
    subj_data = epochs.get_data(units="uV", picks=ch_roi)

    print(">> intercept-only")
    outer_scores_int, best_alphas_int = nested_cv(subj_data, X=X_int,
                                                  alpha_range=alpha_range,
                                                  ssp=False, info=None,
                                                  roi_idx=None,   # not fitting SSP, input data is 6 channels only
                                                  n_splits_outer=5, n_splits_inner=5,
                                                  ridge_tol=1e-4, solver="auto", max_iter=MAX_ITER,
                                                  score_on_mean=False,
                                                  verbose=True)
    outer_scores_int_per_subj[i] = outer_scores_int

    print(">> intercept + surprisal")
    if which_list == 1:
        X = np.hstack([list1_matrix.predictors["intercept"],
                       list1_matrix.predictors["surprisal"]])
    elif which_list == 2:
        X = np.hstack([list2_matrix.predictors["intercept"],
                       list2_matrix.predictors["surprisal"]])
    
    outer_scores_int_logprobs, best_alphas_int_logprobs = nested_cv(subj_data, X=X,
                                                    alpha_range=alpha_range,
                                                    ssp=False, info=None,
                                                    n_splits_outer=5, n_splits_inner=5,
                                                    ridge_tol=1e-4, solver="auto", max_iter=MAX_ITER,
                                                    score_on_mean=False,
                                                    verbose=True)
    outer_scores_int_logprobs_per_subj[i] = outer_scores_int_logprobs
    
    best_alphas_per_subj.append((best_alphas_int, best_alphas_int_logprobs))
    clear_output()
        


In [34]:
delta_r2_per_subj = outer_scores_int_logprobs_per_subj - outer_scores_int_per_subj
delta_r2_per_subj = np.array([d for d in delta_r2_per_subj])                       
delta_r2_per_subj_fold_mean = np.mean(delta_r2_per_subj, axis=1)                   

outer_scores_int_logprobs_per_subj = np.array([s for s in outer_scores_int_logprobs_per_subj])
outer_scores_int_per_subj = np.array([s for s in outer_scores_int_per_subj])
outer_scores_int_logprobs_per_subj_mean = np.mean(np.mean(outer_scores_int_logprobs_per_subj, axis=1), axis=0)
outer_scores_int_per_subj_mean = np.mean(np.mean(outer_scores_int_per_subj, axis=1), axis=0)

improve = [ttest_1samp(delta_r2_per_subj_fold_mean[:, i], 0.0, alternative="greater")
                                           for i in range(delta_r2_per_subj.shape[-1])]
ts = [im.statistic for im in improve]
ps = [im.pvalue for im in improve]
_, qs = fdrcorrection(ps, alpha=0.01, method="indep", is_sorted=False)
improve_df = pd.DataFrame({"R^2 (full)": outer_scores_int_logprobs_per_subj_mean,
                           "R^2 (intercept)": outer_scores_int_per_subj_mean,
                           "delta R^2": np.mean(delta_r2_per_subj_fold_mean, axis=0),
                           "t-statistic": ts, "p-val": ps, "q-val (fdr)": qs})
improve_df = improve_df.rename(index={ch_roi.index(ch): ch for ch in ch_roi})
display(improve_df)

,R^2 (full),R^2 (intercept),delta R^2,t-statistic,p-val,q-val (fdr)
CZ,0.027496,0.028754,-0.001258,-2.462923,0.987954,0.998631
C3,0.021956,0.023254,-0.001298,-2.986966,0.996047,0.998631
C4,0.009811,0.011777,-0.001967,-2.232332,0.980732,0.998631
CP3,0.002502,0.004151,-0.001648,-3.469084,0.998631,0.998631
CP4,0.013160,0.014083,-0.000923,-1.823008,0.957522,0.998631
CPZ,0.019209,0.020656,-0.001447,-2.642568,0.991727,0.998631
